# 01 · The mathematics behind a policy

An independent course by Aditya Gudal.

I · Foundations · 60–120 minutes including exercises.

Read the explanation, work the numbers, edit the code, and explain a failure.

**Runtime:** Run the setup cell in Colab or an installed local Python environment. This notebook embeds its editable Python modules; no curriculum ZIP upload is required. CPU is the default. Robot chapters download pinned source assets separately. See `INSTALL.md` for reproducible local setup and optional Stable Baselines3.

In [ ]:
import sys, os, subprocess, zipfile, json, math, copy, inspect
from pathlib import Path
IN_COLAB='google.colab' in sys.modules
if IN_COLAB:
    # Keep Colab's existing CUDA-enabled Torch; install only course essentials.
    subprocess.check_call([sys.executable,'-m','pip','install','-q','gymnasium[mujoco]==1.4.0','mujoco==3.3.7','matplotlib','imageio','imageio-ffmpeg'])
    os.environ.setdefault('MUJOCO_GL','egl')
elif sys.platform.startswith('linux') and not os.environ.get('DISPLAY'):
    os.environ.setdefault('MUJOCO_GL','egl')
def find_root():
    return next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'lab'/'rl_core.py').exists()),None)
ROOT=find_root()
if ROOT is None:
    # The notebook carries a compressed snapshot of its own editable course modules.
    # No private Site authentication, ZIP upload, GitHub runtime fetch or secret is needed.
    import base64,zlib
    ROOT=Path.cwd()/'Embodied_RL_Notebook';(ROOT/'lab').mkdir(parents=True,exist_ok=True)
    sources=json.loads(zlib.decompress(base64.b64decode('...')))
    for name,source in sources.items():
        if Path(name).name!=name or not name.endswith('.py'):raise ValueError('Invalid embedded module')
        (ROOT/'lab'/name).write_text(source)
    os.chdir(ROOT)
    print('Restored editable course modules to',ROOT/'lab')
sys.path.insert(0,str(ROOT/'lab'))
import numpy as np, torch
from torch import nn
from torch.nn import functional as F
from torch.distributions import Normal,Categorical
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
from IPython.display import display,Markdown,Code,Image
from IPython.core.pylabtools import select_figure_formats
select_figure_formats(get_ipython(),{'png'})
import rl_core as core
from rl_core import mlp,polyak,discounted_returns
from course_figures import sketch
torch.set_num_threads(1);torch.manual_seed(7)
USE_GPU=False  # Set True in Colab if CUDA is actually allocated; CPU is the baseline.
DEVICE='cuda' if USE_GPU and torch.cuda.is_available() else 'cpu'
print('Python',sys.version.split()[0],'Torch',torch.__version__,'selected device:',DEVICE)
print('CPU exercises were executed on Linux. Mac CPU / Colab CUDA paths are provided, not remotely verified.')
def ensure_robot_assets():
    import embodiment_env
    if not (embodiment_env.ASSETS/'franka_panda'/'scene.xml').exists():
        print('Downloading pinned upstream robot assets (~151 MiB) and preparing the passive R1 import.')
        subprocess.check_call([sys.executable,str(ROOT/'lab'/'setup_robot_assets.py')])
    assert (embodiment_env.ASSETS/'galaxea_r1_pro'/'r1_pro_passive.xml').exists()

### A distribution is a way to act, not just a way to describe uncertainty
Imagine a Franka wrist above a cube. One policy always commands the same small motion. Another assigns probabilities to several motions and samples one. Neither policy knows the reward of an untried action. A stochastic policy lets us explore while retaining a mathematical description of exactly how likely each sampled action was. That description becomes the handle used by a policy-gradient update.

Before learning an algorithm, separate three objects: an outcome, its probability, and its value. If a grasp succeeds with probability 0.7 and earns 10, its expected reward is 7; no individual attempt earns 7. An expectation averages over possible outcomes. An empirical mean averages actual samples. The two become close under suitable sampling assumptions, but a small sample can be misleading. Ten successful grasps do not prove certainty.

For discrete actions, a network outputs logits. A softmax converts them into nonnegative numbers that sum to one. Adding the same constant to all logits leaves those probabilities unchanged. For continuous commands, a diagonal Gaussian uses a mean vector and one standard deviation per action coordinate. Its density can exceed one because a density is not the probability of an exact action. Probabilities come from integrals over regions. This matters when comparing log probabilities between discrete and continuous policies.

### Why logarithms appear in an action update
The probability of a trajectory multiplies many terms. Logarithms turn that product into a sum, which is numerically and algebraically easier to differentiate. A derivative asks how a small parameter change affects a quantity. A gradient collects those derivatives for all parameters. PyTorch computes gradients of a scalar loss; it does not know that the loss is an RL objective. A sign error can therefore train a perfectly functional network to make good actions less likely.

We will use vectors for observations and actions, batches for independent samples, and a leading time axis for rollouts. Write tensor shapes before implementing any equation. A Gaussian log density initially has one value per action dimension. Summing over the action axis produces one log probability per transition. Averaging instead changes its scale with the number of joints and silently changes learning behavior between Franka and a bimanual robot.

In [ ]:
fig=sketch('probability');display(fig);plt.close(fig)

### Derivation: from logits to a useful derivative
For logits $z_i$, define $p_i=\exp(z_i)/\sum_j\exp(z_j)$. Then
$$\log p_i=z_i-\log\sum_j\exp(z_j),\qquad
\frac{\partial\log p_i}{\partial z_k}=\mathbf1[i=k]-p_k.$$
The derivative is positive for the sampled action's own logit and negative for alternatives. Multiplying it by a positive advantage pushes probability toward the sampled action; a negative advantage reverses that direction.

For one Gaussian coordinate,
$$\log p(a)=-\frac{(a-\mu)^2}{2\sigma^2}-\log\sigma-\tfrac12\log(2\pi),\qquad
\frac{\partial\log p(a)}{\partial\mu}=\frac{a-\mu}{\sigma^2}.$$
Treat the sampled action as fixed in a score-function update. A reparameterized SAC update is different: $a$ is a differentiable function of the policy parameters. We will make that distinction explicit later.

### Work a gradient by hand
Let the probability of moving right be $p=0.25$. For a sampled right action, the derivative of its log probability with respect to its own logit is $1-p=0.75$. If its advantage is $+2$, the ascent contribution is $+1.5$. If the advantage is $-2$, it becomes $-1.5$. The environment did not supply either gradient: it supplied outcomes from which we built the weighting signal.

In the interactive sketch, change the Gaussian mean and spread. Follow what changes: the likely action, the probability mass inside a region, and the density at a point. These are three different quantities.

In [ ]:
logits=torch.tensor([0.,0.,0.],requires_grad=True)
p=torch.softmax(logits,0); lp=p[1].log(); lp.backward()
expected=torch.tensor([0.,1.,0.])-p.detach()
print('probabilities:',p.detach(),'score gradient:',logits.grad)
assert torch.allclose(logits.grad,expected)
mean=torch.tensor(.2,requires_grad=True);std=.5;fixed_action=torch.tensor(.6)
Normal(mean,std).log_prob(fixed_action).backward()
print('Gaussian analytic / autograd:',(.6-.2)/std**2,mean.grad.item())
assert abs(mean.grad.item()-1.6)<1e-5

In [ ]:
actor=core.SquashedActor(3,7)
obs=torch.zeros(4,3);u,a,lp=actor.sample(obs)
print('obs, latent, action, joint logp shapes:',obs.shape,u.shape,a.shape,lp.shape)
assert lp.shape==(4,)
assert torch.isfinite(actor.log_prob(obs,torch.full((4,7),10.))).all()

## Your experiment

Implement softmax without calling softmax, compare its analytic derivative with autograd, and verify that joint log probabilities sum over the action axis. Then use finite differences to check the Gaussian mean derivative. Repeat with a standard deviation of 0.1 and explain the larger gradient magnitude.

Record your prediction, exact change, measured result, and failure explanation. Longer training is optional and must use a declared budget.

In [ ]:
# Your lab record: edit these fields after making one controlled change.
lab_record={'prediction':'','change':'','result':'','explanation':''}
print(lab_record)

## Explain it back

If you double the number of independent action coordinates, what happens to the joint log probability?

<details><summary>Compare your reasoning</summary>

It is the sum of all coordinate log densities; its magnitude commonly grows with dimension. Entropy and temperature conventions must account for that dimension.

</details>

## Failure to watch for

A probability density is not a probability, and an action-dimension mean is not a joint log probability.

## Primary sources

- https://spinningup.openai.com/en/latest/spinningup/rl_intro.html
- https://pytorch.org/docs/stable/distributions.html
